# 🐘 Hadoop_Ecommerce_Lab – Phân tích doanh thu đơn hàng TMĐT bằng Hadoop Streaming
**Môi trường:** Google Colab (CPU) · Apache Hadoop 3.5.0 · Java 17 · Python Mapper/Reducer
**Dữ liệu:** `orders_2026_09.csv` (1.500 giao dịch do thầy cung cấp)

Notebook này kết hợp **2 tài liệu của thầy**:
| Phần | Nội dung | Theo tài liệu |
|---|---|---|
| **A** | Cài Java 17 + Hadoop 3.5.0, cấu hình | Colab – Bước 2, 3 |
| **B** | Upload dữ liệu, kiểm tra Mapper/Reducer, **chạy Streaming (chế độ local)** → Top 3 | Colab – Bước 4–9 *(phần chính, ổn định)* |
| **C** | **HDFS thật** (NameNode + DataNode), `-put`, `-ls`, `-head`, chạy Streaming trên HDFS, YARN | Colab – Mở rộng + Bài gốc Bước 1–5 *(lấy điểm minh chứng HDFS)* |
| **D** | Đóng gói bài nộp `MSSV_HoTen_HadoopLab`, bản nháp báo cáo | Bài gốc – Sản phẩm đầu ra |

**Cách dùng:** chạy từng ô từ trên xuống (Shift+Enter), ô trước xong mới chạy ô sau. Ô có 📸 = **chụp màn hình** làm minh chứng.
Nếu Colab bị ngắt kết nối: chạy lại từ **Ô 1** (file trong `/content` sẽ mất).

# PHẦN A – Dựng môi trường (Colab Bước 2–3)

### Ô 1 · Cài Java 17 và tải Hadoop 3.5.0 (mất 3–8 phút)
Hadoop 3.5 cần **Java 17**. Nếu ô báo lỗi tải file, bấm chạy lại.

In [1]:
%%bash
set -e
echo "Java mac dinh cua Colab:"; java -version 2>&1 | head -1
apt-get update -qq
apt-get install -y -qq openjdk-17-jdk-headless > /dev/null
cd /content
if [ ! -d /content/hadoop-3.5.0 ]; then
  if [ ! -f hadoop-3.5.0.tar.gz ]; then
    wget -q https://archive.apache.org/dist/hadoop/common/hadoop-3.5.0/hadoop-3.5.0.tar.gz \
     || { rm -f hadoop-3.5.0.tar.gz; wget -q https://downloads.apache.org/hadoop/common/hadoop-3.5.0/hadoop-3.5.0.tar.gz; }
  fi
  tar -xzf hadoop-3.5.0.tar.gz || { rm -f hadoop-3.5.0.tar.gz; echo "FILE TAI BI LOI -> chay lai o nay"; exit 1; }
fi
echo "=== CAI DAT XONG ==="
ls -d /content/hadoop-3.5.0
ls /usr/lib/jvm/

Java mac dinh cua Colab:
[0.016s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
=== CAI DAT XONG ===
/content/hadoop-3.5.0
java-1.17.0-openjdk-amd64
java-1.21.0-openjdk-amd64
java-17-openjdk-amd64
java-21-openjdk-amd64


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


### Ô 2 · Cấu hình biến môi trường + `hadoop version`  📸
Kết quả đúng: dòng đầu in ra **`Hadoop 3.5.0`**. Ô này cũng định nghĩa hàm `run()` dùng ở các ô sau (nên **phải chạy ô này trước**).

In [2]:
import os, subprocess

HADOOP_HOME = "/content/hadoop-3.5.0"
JAVA_HOME = "/usr/lib/jvm/java-17-openjdk-amd64"
assert os.path.isdir(JAVA_HOME), "Chua co Java 17 - chay lai O 1"
assert os.path.isdir(HADOOP_HOME), "Chua co Hadoop - chay lai O 1"

os.environ.update({
    "HADOOP_HOME": HADOOP_HOME,
    "JAVA_HOME": JAVA_HOME,
    "HDFS_NAMENODE_USER": "root",
    "HDFS_DATANODE_USER": "root",
    "HDFS_SECONDARYNAMENODE_USER": "root",
    "YARN_RESOURCEMANAGER_USER": "root",
    "YARN_NODEMANAGER_USER": "root",
})
for p in (HADOOP_HOME + "/bin", HADOOP_HOME + "/sbin", JAVA_HOME + "/bin"):
    if p not in os.environ["PATH"].split(":"):
        os.environ["PATH"] = p + ":" + os.environ["PATH"]

envfile = HADOOP_HOME + "/etc/hadoop/hadoop-env.sh"
line = "export JAVA_HOME=" + JAVA_HOME + "\n"
if line not in open(envfile).read():
    open(envfile, "a").write("\n" + line)

os.makedirs("/content/logs", exist_ok=True)

def run(cmd, tail=60, check=True, cwd="/content", log=None):
    """Chay lenh, in ra phan cuoi cua ket qua (de chup man hinh), luu log day du vao /content/logs."""
    shell = isinstance(cmd, str)
    r = subprocess.run(cmd, shell=shell, cwd=cwd, capture_output=True, text=True,
                       executable="/bin/bash" if shell else None)
    out = ((r.stdout or "") + (r.stderr or "")).rstrip()
    if log:
        open(f"/content/logs/{log}.txt", "w").write(out + "\n")
    lines = out.splitlines()
    if len(lines) > tail:
        print(f"... (bo bot {len(lines)-tail} dong dau, log day du trong /content/logs/{log or '...'}.txt)")
    print("\n".join(lines[-tail:]))
    if check and r.returncode != 0:
        raise RuntimeError(f"Lenh that bai (ma loi {r.returncode})")
    return r

print("JAVA_HOME  =", os.environ["JAVA_HOME"])
print("HADOOP_HOME =", os.environ["HADOOP_HOME"])
print()
run("hadoop version", tail=8, log="01_hadoop_version")

JAVA_HOME  = /usr/lib/jvm/java-17-openjdk-amd64
HADOOP_HOME = /content/hadoop-3.5.0

Hadoop 3.5.0
Source code repository https://github.com/apache/hadoop.git -r dbcc7cd797100e6b32cd84f85b53a5193a5f9af0
Compiled by cnauroth on 2026-03-24T16:56Z
Compiled on platform linux-x86_64
Compiled with protoc 3.25.5
From source with checksum 64582ed0a62a169c39a19c734a8f33b5
This command was run using /content/hadoop-3.5.0/share/hadoop/common/hadoop-common-3.5.0.jar


CompletedProcess(args='hadoop version', returncode=0, stdout='Hadoop 3.5.0\nSource code repository https://github.com/apache/hadoop.git -r dbcc7cd797100e6b32cd84f85b53a5193a5f9af0\nCompiled by cnauroth on 2026-03-24T16:56Z\nCompiled on platform linux-x86_64\nCompiled with protoc 3.25.5\nFrom source with checksum 64582ed0a62a169c39a19c734a8f33b5\nThis command was run using /content/hadoop-3.5.0/share/hadoop/common/hadoop-common-3.5.0.jar\n', stderr='')

# PHẦN B – Chạy MapReduce (Hadoop Streaming, chế độ local) – Colab Bước 4–9

### Ô 3 · Upload 3 file của thầy
Bấm **Choose Files**, chọn **cùng lúc 3 file**: `orders_2026_09.csv`, `mapper_revenue.py`, `reducer_revenue.py`.

In [3]:
os.chdir("/content")
from google.colab import files
uploaded = files.upload()
print("Da upload:", list(uploaded.keys()))
for f in ("orders_2026_09.csv","mapper_revenue.py","reducer_revenue.py"):
    assert os.path.exists("/content/"+f), "Thieu file: " + f
print("✅ Du 3 file")

Saving mapper_revenue.py to mapper_revenue (1).py
Saving reducer_revenue.py to reducer_revenue (1).py
Saving orders_2026_09.csv to orders_2026_09 (1).csv
Da upload: ['mapper_revenue (1).py', 'reducer_revenue (1).py', 'orders_2026_09 (1).csv']
✅ Du 3 file


### Ô 4 · Kiểm tra nhanh bằng **dữ liệu mẫu 5 dòng của thầy** (Colab Bước 7)
Kết quả mong đợi đúng như slide của thầy: **Book 250000 · Electronics 5900000 · Fashion 520000**. (Ô này dùng file riêng `orders_sample.csv`, không đụng đến file dữ liệu thật.)

In [4]:
sample = """transaction_id,order_date,order_time,province,category,channel,quantity,unit_price,amount,status,shipping_days
TX001,2026-09-01,08:30,Ha Noi,Electronics,Web,1,3400000,3400000,SUCCESS,2
TX002,2026-09-01,09:00,Da Nang,Fashion,App,2,260000,520000,SUCCESS,3
TX003,2026-09-02,10:30,Can Tho,Grocery,Web,1,180000,180000,FAILED,4
TX004,2026-09-02,11:00,Lam Dong,Book,App,1,250000,250000,SUCCESS,2
TX005,2026-09-03,13:30,Ha Noi,Electronics,Web,1,2500000,2500000,SUCCESS,1
"""
open("/content/orders_sample.csv","w").write(sample)
r = run("cat orders_sample.csv | python3 mapper_revenue.py | sort | python3 reducer_revenue.py", tail=20)
got = dict(l.split("\t") for l in r.stdout.strip().splitlines())
print("✅ DUNG – khop slide cua thay" if got == {"Book":"250000","Electronics":"5900000","Fashion":"520000"} else "❌ SAI – kiem tra mapper/reducer")

Book	250000
Electronics	5900000
Fashion	520000
✅ DUNG – khop slide cua thay


### Ô 5 · Chạy thử pipeline trên **dữ liệu thật** (1.500 dòng)
Kết quả mong đợi: Electronics **3703769311**, Home 410315065, Fashion 218597763, ...

In [5]:
run("wc -l orders_2026_09.csv && head -3 orders_2026_09.csv", tail=5)
print("\n=== mapper | sort | reducer ===")
run("cat orders_2026_09.csv | python3 mapper_revenue.py | sort | python3 reducer_revenue.py", tail=20)

1501 orders_2026_09.csv
transaction_id,order_date,order_time,province,category,channel,quantity,unit_price,amount,status,shipping_days
TX000001,2026-09-30,02:25:00,Ha Noi,Beauty,web,1,294947,294947,SUCCESS,2
TX000002,2026-09-23,23:58:00,Da Nang,Grocery,mobile,1,265514,265514,SUCCESS,3

=== mapper | sort | reducer ===
Beauty	111032003
Book	39306022
Electronics	3703769311
Fashion	218597763
Grocery	80983147
Home	410315065
Sport	166194354
Toy	83881779


CompletedProcess(args='cat orders_2026_09.csv | python3 mapper_revenue.py | sort | python3 reducer_revenue.py', returncode=0, stdout='Beauty\t111032003\nBook\t39306022\nElectronics\t3703769311\nFashion\t218597763\nGrocery\t80983147\nHome\t410315065\nSport\t166194354\nToy\t83881779\n', stderr='')

### Ô 6 · Chạy Hadoop Streaming ở chế độ local (Colab Bước 8)  📸
Chụp **lệnh** (dòng `LENH CHAY:`) và **phần log cuối** (có `Job job_local... completed successfully`).

In [6]:
import glob, shutil
shutil.rmtree("/content/output_revenue", ignore_errors=True)
jar = glob.glob(HADOOP_HOME + "/share/hadoop/tools/lib/hadoop-streaming*.jar")[0]

cmd = [
    "hadoop", "jar", jar,
    "-D", "mapreduce.framework.name=local",
    "-D", "fs.defaultFS=file:///",
    "-D", "mapreduce.job.reduces=1",
    "-files", "/content/mapper_revenue.py,/content/reducer_revenue.py",
    "-mapper", "python3 mapper_revenue.py",
    "-reducer", "python3 reducer_revenue.py",
    "-input", "file:///content/orders_2026_09.csv",
    "-output", "file:///content/output_revenue",
]
print("LENH CHAY:\n", " ".join(cmd), "\n")
run(cmd, tail=45, log="02_streaming_local")

LENH CHAY:
 hadoop jar /content/hadoop-3.5.0/share/hadoop/tools/lib/hadoop-streaming-3.5.0.jar -D mapreduce.framework.name=local -D fs.defaultFS=file:/// -D mapreduce.job.reduces=1 -files /content/mapper_revenue.py,/content/reducer_revenue.py -mapper python3 mapper_revenue.py -reducer python3 reducer_revenue.py -input file:///content/orders_2026_09.csv -output file:///content/output_revenue 

... (bo bot 152 dong dau, log day du trong /content/logs/02_streaming_local.txt)
		WRONG_REDUCE=0
	File Output Format Counters 
		Bytes Written=145
2026-10-10 09:43:12,910 INFO mapred.LocalJobRunner: Finishing task: attempt_local1961660370_0001_r_000000_0
2026-10-10 09:43:12,911 INFO mapred.LocalJobRunner: reduce task executor complete.
2026-10-10 09:43:13,018 INFO mapreduce.Job: Job job_local1961660370_0001 running in uber mode : false
2026-10-10 09:43:13,019 INFO mapreduce.Job:  map 100% reduce 100%
2026-10-10 09:43:13,021 INFO mapreduce.Job: Job job_local1961660370_0001 completed successfully
2

CompletedProcess(args=['hadoop', 'jar', '/content/hadoop-3.5.0/share/hadoop/tools/lib/hadoop-streaming-3.5.0.jar', '-D', 'mapreduce.framework.name=local', '-D', 'fs.defaultFS=file:///', '-D', 'mapreduce.job.reduces=1', '-files', '/content/mapper_revenue.py,/content/reducer_revenue.py', '-mapper', 'python3 mapper_revenue.py', '-reducer', 'python3 reducer_revenue.py', '-input', 'file:///content/orders_2026_09.csv', '-output', 'file:///content/output_revenue'], returncode=0, stdout='', stderr="2026-10-10 09:43:10,985 WARN impl.MetricsSystemImpl: JobTracker metrics system already initialized!\n2026-10-10 09:43:11,197 INFO mapred.FileInputFormat: Total input files to process : 1\n2026-10-10 09:43:11,220 INFO mapreduce.JobSubmitter: number of splits:1\n2026-10-10 09:43:11,490 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local1961660370_0001\n2026-10-10 09:43:11,491 INFO mapreduce.JobSubmitter: Executing with tokens: []\n2026-10-10 09:43:11,858 INFO mapred.LocalDistributedCache

### Ô 7 · Xem kết quả và lấy Top 3 (Colab Bước 9)  📸
Chụp `ls -l` (có `part-00000` và `_SUCCESS`), nội dung `part-*` và Top 3.

In [7]:
print("=== ls -l /content/output_revenue ===")
run("ls -l /content/output_revenue", tail=10)
print("\n=== cat part-* ===")
run("cat /content/output_revenue/part-*", tail=20)
print("\n=== TOP 3 (sap xep giam dan) ===")
run("sort -k2,2nr /content/output_revenue/part-* | head -3", tail=5)

=== ls -l /content/output_revenue ===
total 4
-rw-r--r-- 1 root root 133 Oct 10 09:43 part-00000
-rw-r--r-- 1 root root   0 Oct 10 09:43 _SUCCESS

=== cat part-* ===
Beauty	111032003
Book	39306022
Electronics	3703769311
Fashion	218597763
Grocery	80983147
Home	410315065
Sport	166194354
Toy	83881779

=== TOP 3 (sap xep giam dan) ===
Electronics	3703769311
Home	410315065
Fashion	218597763


CompletedProcess(args='sort -k2,2nr /content/output_revenue/part-* | head -3', returncode=0, stdout='Electronics\t3703769311\nHome\t410315065\nFashion\t218597763\n', stderr='')

### Ô 8 · Kiểm chứng: so sánh với Pandas
Dòng cuối phải là `✅ KHOP`.

In [8]:
import pandas as pd
df = pd.read_csv("/content/orders_2026_09.csv")
expected = df[df.status == "SUCCESS"].groupby("category").amount.sum().to_dict()
got = {}
for ln in open("/content/output_revenue/part-00000"):
    k, v = ln.rstrip("\n").split("\t"); got[k] = int(v)
print(f"Tong giao dich: {len(df)} | SUCCESS: {(df.status=='SUCCESS').sum()} | FAILED: {(df.status=='FAILED').sum()} | RETURNED: {(df.status=='RETURNED').sum()}")
print("✅ KHOP – ket qua MapReduce dung voi Pandas" if got == expected else "❌ KHAC NHAU – kiem tra lai")

Tong giao dich: 1500 | SUCCESS: 1240 | FAILED: 142 | RETURNED: 118
✅ KHOP – ket qua MapReduce dung voi Pandas


# PHẦN C – HDFS thật + YARN (Colab "Mở rộng" + Bài gốc Bước 1–5)
Bài gốc chấm **2 điểm** cho việc "upload dữ liệu vào HDFS" nên nên làm phần này. Thầy lưu ý phần này **có thể lỗi** do Colab là máy ảo tạm thời; nếu lỗi, bạn vẫn đã hoàn thành phần chính ở Phần B.

### Ô 9 · Cấu hình HDFS (Colab Mở rộng – cấu hình XML)

In [9]:
os.makedirs("/content/hadoopdata/namenode", exist_ok=True)
os.makedirs("/content/hadoopdata/datanode", exist_ok=True)
conf = HADOOP_HOME + "/etc/hadoop/"
open(conf+"core-site.xml","w").write("""<configuration>
  <property><name>fs.defaultFS</name><value>hdfs://localhost:9000</value></property>
</configuration>""")
open(conf+"hdfs-site.xml","w").write("""<configuration>
  <property><name>dfs.replication</name><value>1</value></property>
  <property><name>dfs.namenode.name.dir</name><value>file:/content/hadoopdata/namenode</value></property>
  <property><name>dfs.datanode.data.dir</name><value>file:/content/hadoopdata/datanode</value></property>
</configuration>""")
print("✅ Da ghi core-site.xml va hdfs-site.xml")

✅ Da ghi core-site.xml va hdfs-site.xml


### Ô 10 · Format NameNode và khởi động HDFS  📸
Dùng `hdfs --daemon start` (không cần SSH nên ổn định hơn `start-dfs.sh`). Kết quả đúng: `jps` có **NameNode** và **DataNode**; `hdfs dfsadmin -report` báo `Live datanodes (1)`.

In [10]:
%%bash
pkill -f proc_namenode 2>/dev/null || true
pkill -f proc_datanode 2>/dev/null || true
sleep 2
hdfs namenode -format -force > /content/logs/03_format.txt 2>&1 && echo "Format NameNode: OK"
hdfs --daemon start namenode
hdfs --daemon start datanode
sleep 12
hdfs dfsadmin -safemode wait
echo
echo "=== jps ==="; jps
echo
echo "=== hdfs dfsadmin -report (rut gon) ==="
hdfs dfsadmin -report | grep -E "Live datanodes|Configured Capacity|DFS Used:" | head -4

Format NameNode: OK
Safe mode is OFF

=== jps ===
4226 NameNode
4534 Jps
4301 DataNode

=== hdfs dfsadmin -report (rut gon) ===
Configured Capacity: 120942624768 (112.64 GB)
DFS Used: 24576 (24 KB)
Live datanodes (1):
Configured Capacity: 120942624768 (112.64 GB)


### Ô 11 · Bước 1 bài gốc: Tạo thư mục và đưa dữ liệu lên HDFS  📸 **(Minh chứng 01)**
Chụp phần kết quả `-ls` và `-head`.

In [11]:
%%bash
cd /content
hadoop fs -mkdir -p /user/student/ecommerce/input
hadoop fs -mkdir -p /user/student/ecommerce/output
hadoop fs -put -f orders_2026_09.csv /user/student/ecommerce/input/
echo "=== hadoop fs -ls /user/student/ecommerce/input ==="
hadoop fs -ls /user/student/ecommerce/input
echo
echo "=== hadoop fs -head ... (5 dong dau) ==="
hadoop fs -head /user/student/ecommerce/input/orders_2026_09.csv | head -5

=== hadoop fs -ls /user/student/ecommerce/input ===
Found 1 items
-rw-r--r--   1 root supergroup     121497 2026-10-10 09:43 /user/student/ecommerce/input/orders_2026_09.csv

=== hadoop fs -head ... (5 dong dau) ===
transaction_id,order_date,order_time,province,category,channel,quantity,unit_price,amount,status,shipping_days
TX000001,2026-09-30,02:25:00,Ha Noi,Beauty,web,1,294947,294947,SUCCESS,2
TX000002,2026-09-23,23:58:00,Da Nang,Grocery,mobile,1,265514,265514,SUCCESS,3
TX000003,2026-09-05,15:38:00,Ha Noi,Electronics,partner,1,12741238,12741238,SUCCESS,3
TX000004,2026-09-09,18:01:00,Hai Phong,Electronics,mobile,1,8846997,8846997,SUCCESS,4


### Ô 12 · Bước 3 bài gốc: Chạy Hadoop Streaming với input/output trên HDFS  📸
Chụp lệnh và log cuối.

In [12]:
run("hadoop fs -rm -r -f /user/student/ecommerce/output/revenue_by_category", tail=3)
cmd = [
    "hadoop", "jar", jar,
    "-D", "mapreduce.job.reduces=1",
    "-files", "mapper_revenue.py,reducer_revenue.py",
    "-mapper", "python3 mapper_revenue.py",
    "-reducer", "python3 reducer_revenue.py",
    "-input", "/user/student/ecommerce/input/orders_2026_09.csv",
    "-output", "/user/student/ecommerce/output/revenue_by_category",
]
print("LENH CHAY:\n", " ".join(cmd), "\n")
run(cmd, tail=45, log="04_streaming_hdfs")


LENH CHAY:
 hadoop jar /content/hadoop-3.5.0/share/hadoop/tools/lib/hadoop-streaming-3.5.0.jar -D mapreduce.job.reduces=1 -files mapper_revenue.py,reducer_revenue.py -mapper python3 mapper_revenue.py -reducer python3 reducer_revenue.py -input /user/student/ecommerce/input/orders_2026_09.csv -output /user/student/ecommerce/output/revenue_by_category 

... (bo bot 170 dong dau, log day du trong /content/logs/04_streaming_hdfs.txt)
2026-10-10 09:44:04,154 INFO mapreduce.Job:  map 100% reduce 100%
2026-10-10 09:44:05,162 INFO mapreduce.Job: Job job_local72620724_0001 completed successfully
2026-10-10 09:44:05,171 INFO mapreduce.Job: Counters: 36
	File System Counters
		FILE: Number of bytes read=335444
		FILE: Number of bytes written=1824732
		FILE: Number of read operations=0
		FILE: Number of large read operations=0
		FILE: Number of write operations=0
		HDFS: Number of bytes read=242994
		HDFS: Number of bytes written=133
		HDFS: Number of read operations=15
		HDFS: Number of large rea

CompletedProcess(args=['hadoop', 'jar', '/content/hadoop-3.5.0/share/hadoop/tools/lib/hadoop-streaming-3.5.0.jar', '-D', 'mapreduce.job.reduces=1', '-files', 'mapper_revenue.py,reducer_revenue.py', '-mapper', 'python3 mapper_revenue.py', '-reducer', 'python3 reducer_revenue.py', '-input', '/user/student/ecommerce/input/orders_2026_09.csv', '-output', '/user/student/ecommerce/output/revenue_by_category'], returncode=0, stdout='', stderr="2026-10-10 09:44:01,973 WARN impl.MetricsSystemImpl: JobTracker metrics system already initialized!\n2026-10-10 09:44:02,374 INFO mapred.FileInputFormat: Total input files to process : 1\n2026-10-10 09:44:02,448 INFO mapreduce.JobSubmitter: number of splits:1\n2026-10-10 09:44:02,710 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local72620724_0001\n2026-10-10 09:44:02,712 INFO mapreduce.JobSubmitter: Executing with tokens: []\n2026-10-10 09:44:03,003 INFO mapred.LocalDistributedCacheManager: Localized file:/content/mapper_revenue.py as fil

### Ô 13 · Bước 4 bài gốc: Xem kết quả trên HDFS và lấy Top 3  📸 **(Minh chứng 02)**
Chụp `-ls` (có `part-00000`), `-cat` và Top 3.

In [13]:
%%bash
cd /content
echo "=== hadoop fs -ls output ==="
hadoop fs -ls /user/student/ecommerce/output/revenue_by_category
echo
echo "=== hadoop fs -cat output/part-* ==="
hadoop fs -cat /user/student/ecommerce/output/revenue_by_category/part-*
hadoop fs -get -f /user/student/ecommerce/output/revenue_by_category/part-* ./result_revenue_by_category.txt
echo
echo "=== TOP 3 ==="
sort -k2,2nr result_revenue_by_category.txt | head -3

=== hadoop fs -ls output ===
Found 2 items
-rw-r--r--   1 root supergroup          0 2026-10-10 09:44 /user/student/ecommerce/output/revenue_by_category/_SUCCESS
-rw-r--r--   1 root supergroup        133 2026-10-10 09:44 /user/student/ecommerce/output/revenue_by_category/part-00000

=== hadoop fs -cat output/part-* ===
Beauty	111032003
Book	39306022
Electronics	3703769311
Fashion	218597763
Grocery	80983147
Home	410315065
Sport	166194354
Toy	83881779

=== TOP 3 ===
Electronics	3703769311
Home	410315065
Fashion	218597763


### Ô 14 · Đối chiếu: kết quả HDFS có giống kết quả local (Phần B) không?

In [14]:
a = open("/content/output_revenue/part-00000").read().split("\n")
b = open("/content/result_revenue_by_category.txt").read().split("\n")
print("✅ GIONG NHAU – local va HDFS cho cung ket qua" if sorted(a) == sorted(b) else "❌ KHAC NHAU")

✅ GIONG NHAU – local va HDFS cho cung ket qua


### Ô 15 · (Tùy chọn) Bước 5 bài gốc: Bật YARN và quan sát job  📸
Cho phần "liên hệ YARN". **Không bắt buộc** – nếu lỗi, bỏ qua và ghi chú trong báo cáo: *"MapReduce chạy ở chế độ local, YARN được giải thích ở mức khái niệm"*. Chạy **2 ô liên tiếp** bên dưới.

In [15]:
conf = HADOOP_HOME + "/etc/hadoop/"
open(conf+"mapred-site.xml","w").write(f"""<configuration>
  <property><name>mapreduce.framework.name</name><value>yarn</value></property>
  <property><name>yarn.app.mapreduce.am.env</name><value>HADOOP_MAPRED_HOME={HADOOP_HOME}</value></property>
  <property><name>mapreduce.map.env</name><value>HADOOP_MAPRED_HOME={HADOOP_HOME}</value></property>
  <property><name>mapreduce.reduce.env</name><value>HADOOP_MAPRED_HOME={HADOOP_HOME}</value></property>
  <property><name>mapreduce.application.classpath</name>
    <value>{HADOOP_HOME}/share/hadoop/mapreduce/*:{HADOOP_HOME}/share/hadoop/mapreduce/lib/*</value></property>
</configuration>""")
open(conf+"yarn-site.xml","w").write("""<configuration>
  <property><name>yarn.resourcemanager.hostname</name><value>localhost</value></property>
  <property><name>yarn.nodemanager.aux-services</name><value>mapreduce_shuffle</value></property>
  <property><name>yarn.nodemanager.vmem-check-enabled</name><value>false</value></property>
  <property><name>yarn.nodemanager.env-whitelist</name>
    <value>JAVA_HOME,HADOOP_COMMON_HOME,HADOOP_HDFS_HOME,HADOOP_CONF_DIR,CLASSPATH_PREPEND_DISTCACHE,HADOOP_YARN_HOME,HADOOP_HOME,HADOOP_MAPRED_HOME,PATH,LANG,TZ</value></property>
</configuration>""")
print("✅ Da ghi cau hinh YARN")

✅ Da ghi cau hinh YARN


In [16]:
%%bash
cd /content
yarn --daemon start resourcemanager
yarn --daemon start nodemanager
sleep 30
echo "=== jps ==="; jps
echo; echo "=== yarn node -list ==="; yarn node -list 2>&1 | tail -4

hadoop fs -rm -r -f /user/student/ecommerce/output/revenue_by_category_yarn
hadoop jar $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming*.jar \
  -D mapreduce.job.reduces=1 \
  -files mapper_revenue.py,reducer_revenue.py \
  -mapper "python3 mapper_revenue.py" \
  -reducer "python3 reducer_revenue.py" \
  -input /user/student/ecommerce/input/orders_2026_09.csv \
  -output /user/student/ecommerce/output/revenue_by_category_yarn > /content/logs/05_streaming_yarn.txt 2>&1
tail -15 /content/logs/05_streaming_yarn.txt

echo; echo "=== yarn application -list -appStates ALL ==="
yarn application -list -appStates ALL 2>&1 | tail -5
echo; echo "=== Ket qua chay tren YARN ==="
hadoop fs -cat /user/student/ecommerce/output/revenue_by_category_yarn/part-*

=== jps ===
4226 NameNode
5351 ResourceManager
5866 Jps
5419 NodeManager
4301 DataNode

=== yarn node -list ===
2026-10-10 09:44:50,460 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at localhost/127.0.0.1:8032
Total Nodes:1
         Node-Id	     Node-State	Node-Http-Address	Number-of-Running-Containers
cc818b37241c:35091	        RUNNING	cc818b37241c:8042	                           0
		Peak Map Virtual memory (bytes)=2746003456
		Peak Reduce Physical memory (bytes)=188989440
		Peak Reduce Virtual memory (bytes)=2749632512
	Shuffle Errors
		BAD_ID=0
		CONNECTION=0
		IO_ERROR=0
		WRONG_LENGTH=0
		WRONG_MAP=0
		WRONG_REDUCE=0
	File Input Format Counters 
		Bytes Read=125593
	File Output Format Counters 
		Bytes Written=130
2026-10-10 09:45:29,808 INFO streaming.StreamJob: Output directory: /user/student/ecommerce/output/revenue_by_category_yarn

=== yarn application -list -appStates ALL ===
2026-10-10 09:45:31,462 INFO client.DefaultNoHARMFailoverProxyProvid

# PHẦN D – Đóng gói bài nộp

### Ô 16 · Tạo thư mục nộp bài, bản nháp báo cáo và tải về
**Sửa tên** `MSSV_HoTen_HadoopLab` thành mã số sinh viên + họ tên của bạn (ví dụ `2012345_NguyenVanA_HadoopLab`) rồi chạy. Ô này tự tạo: bảng Top 3, **bản nháp báo cáo** (đã điền sẵn số liệu thật), các file log, và nén thành `.zip`.

In [17]:
import os, shutil, glob, pandas as pd
FOLDER = "MSSV_HoTen_HadoopLab"      # <<< SUA TEN O DAY

root = f"/content/{FOLDER}"
shutil.rmtree(root, ignore_errors=True)
os.makedirs(root + "/logs", exist_ok=True)

# --- ket qua ---
res = pd.read_csv("/content/output_revenue/part-00000", sep="\t", header=None,
                  names=["Category","Doanh thu"]).sort_values("Doanh thu", ascending=False).reset_index(drop=True)
df = pd.read_csv("/content/orders_2026_09.csv")
n_all, n_ok = len(df), int((df.status=="SUCCESS").sum())
n_fail, n_ret = int((df.status=="FAILED").sum()), int((df.status=="RETURNED").sum())
total = int(res["Doanh thu"].sum())

shutil.copy("/content/output_revenue/part-00000", root + "/part-00000_local.txt")
if os.path.exists("/content/result_revenue_by_category.txt"):
    shutil.copy("/content/result_revenue_by_category.txt", root + "/part-00000_hdfs.txt")
for f in ("mapper_revenue.py","reducer_revenue.py"):
    shutil.copy("/content/"+f, root)
for f in glob.glob("/content/logs/*.txt"):
    shutil.copy(f, root + "/logs")
res.to_csv(root + "/revenue_sorted.csv", index=False)

# --- bang Top 3 ---
rows = ["| Hang | Category | Doanh thu (VND) | Ty trong | Nhan xet |", "|---|---|---|---|---|"]
for i, r in res.head(3).iterrows():
    rows.append(f"| {i+1} | {r['Category']} | {r['Doanh thu']:,} | {r['Doanh thu']/total:.1%} | (tu viet) |")
top3 = "\n".join(rows)
open(root + "/top3.md", "w").write(top3 + "\n")

# --- ban nhap bao cao (so lieu that, phan nhan xet tu viet lai) ---
t1, t2, t3 = res.iloc[0], res.iloc[1], res.iloc[2]
draft = f"""# BAO CAO THUC HANH HADOOP – PHAN TICH DOANH THU THEO NHOM SAN PHAM
Sinh vien: ......  MSSV: ......
Moi truong: Google Colab, Apache Hadoop 3.5.0, Java 17, Hadoop Streaming (Python). Che do local MapReduce; HDFS pseudo-distributed (NameNode + DataNode) tren cung may ao.

## 1. Du lieu dau vao
File orders_2026_09.csv gom {n_all:,} giao dich thang 09/2026, 11 truong (transaction_id, order_date, order_time, province, category, channel, quantity, unit_price, amount, status, shipping_days). Trong do {n_ok:,} giao dich SUCCESS, {n_fail:,} FAILED, {n_ret:,} RETURNED. Chi giao dich SUCCESS duoc tinh doanh thu. Tren HDFS file nam o /user/student/ecommerce/input/.

## 2. Mapper
Mapper doc tung dong CSV tu standard input. Neu status = SUCCESS thi phat sinh cap key-value: category -> amount (vi du: {t1['Category']} -> mot gia tri amount). Cac giao dich FAILED/RETURNED bi bo qua.

## 3. Reducer
Sau buoc Shuffle/Sort, cac dong cung category duoc gom lai. Reducer cong tong amount cua tung category va ghi ket qua cuoi vao part-00000.

## 4. Ket qua Top 3
{top3}

Tong doanh thu SUCCESS: {total:,} VND.

## 5. Nhan xet kinh doanh (5-7 dong – TU VIET LAI BANG LOI CUA EM)
Goi y y tuong: {t1['Category']} dan dau ({t1['Doanh thu']:,} VND, chiem {t1['Doanh thu']/total:.1%} tong doanh thu), gap {t1['Doanh thu']/t2['Doanh thu']:.1f} lan nhom thu hai ({t2['Category']}); nhom thu ba la {t3['Category']}. Phan tich: rui ro phu thuoc vao mot nhom, de xuat ton kho/khuyen mai/giao hang, luu y da loai FAILED va RETURNED.

## 6. Lien he HDFS – MapReduce – YARN
HDFS luu tru phan tan du lieu theo cay thu muc; MapReduce xu ly theo mo hinh key-value (Map -> Shuffle/Sort -> Reduce); YARN quan ly tai nguyen va lap lich cac task cua job.

## 7. Han che cua xu ly batch
(Viet 1-2 cau: ket qua chi co sau khi job chay xong, khong phu hop nhu cau thoi gian thuc, job co chi phi khoi dong...)
"""
open(root + "/bao_cao_nhap.md", "w").write(draft)

print("Ket qua day du (giam dan):"); print(res.to_string(index=False))
print("\nBang Top 3:\n" + top3)
print("\nNoi dung thu muc nop:"); print("\n".join(sorted(os.listdir(root))))

shutil.make_archive(root, "zip", root)
print(f"\n📦 Da tao {root}.zip  →  nho them anh chup man hinh + bao cao PDF truoc khi nop")
from google.colab import files
files.download(root + ".zip")

Ket qua day du (giam dan):
   Category  Doanh thu
Electronics 3703769311
       Home  410315065
    Fashion  218597763
      Sport  166194354
     Beauty  111032003
        Toy   83881779
    Grocery   80983147
       Book   39306022

Bang Top 3:
| Hang | Category | Doanh thu (VND) | Ty trong | Nhan xet |
|---|---|---|---|---|
| 1 | Electronics | 3,703,769,311 | 76.9% | (tu viet) |
| 2 | Home | 410,315,065 | 8.5% | (tu viet) |
| 3 | Fashion | 218,597,763 | 4.5% | (tu viet) |

Noi dung thu muc nop:
bao_cao_nhap.md
logs
mapper_revenue.py
part-00000_hdfs.txt
part-00000_local.txt
reducer_revenue.py
revenue_sorted.csv
top3.md

📦 Da tao /content/MSSV_HoTen_HadoopLab.zip  →  nho them anh chup man hinh + bao cao PDF truoc khi nop


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Ô 17 · Lưu lại công sức
- **Tải notebook kèm kết quả:** *File → Download → Download .ipynb*
- Hoặc *File → Save a copy in Drive*
(File trong `/content` sẽ mất khi phiên Colab kết thúc.)